# Unix-terminal. SSH

## Мотивация

SSH — не только «терминал на другом компьютере», а основа удалённой разработки. VS Code Remote SSH, удалённые IDE JetBrains, передача файлов, синхронизация проекта и доступ к Jupyter на сервере так или иначе используют SSH-соединение. Освоив его, можно работать на мощной машине из привычного редактора, переносить данные, оставлять долгие процессы и безопасно открывать закрытые сервисы. Ошибка в настройках SSH легко лишает доступа, поэтому соединение нужно уметь настраивать и проверять самостоятельно.

Подключаться к учебному серверу вы уже умеете — это была домашка №1. Сегодня разбираем то, что превращает разовое подключение в рабочий инструмент: ключи вместо пароля, `~/.ssh/config`, `ssh-agent`, копирование данных и туннели.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>


До SSH удалённо работали через `telnet` и `rlogin`: пароль уходил по сети открытым текстом, и любой, кто слушал трафик по пути, читал его как обычную строку. В 1995 году в Хельсинкском технологическом университете поймали ровно такую атаку — снифер собирал чужие пароли. Разбирался с ней Тату Улёнен, и по итогам написал первую версию SSH. Позже протокол переработали (SSH-2), а команда OpenBSD сделала из свободной версии OpenSSH — тот самый `ssh`, который стоит сейчас практически везде.

Отсюда и привычка курса: пароль — временная мера для самого первого входа, постоянный доступ настраивают по ключам.


</details>

> **Как устроен семинар.** Все команды выполняются **на вашей виртуальной
> машине, в домашнем каталоге** — в `~/seminar-08/`. Каталог `/tmp` не
> используем: он вычищается при перезагрузке, а ключи и конфиги нужны вам и
> после занятия.
>
> Где в примерах написано `course` — имеется в виду **учебный сервер** из
> домашки №1 под коротким именем из вашего `~/.ssh/config` (раздел 3). Ячейки,
> которым нужен живой сервер, помечены комментарием в первой строке: без
> доступа к серверу они не выполнятся, и это нормально — смотрите, что делает
> преподаватель, и повторите потом.
>
> Свёрнутые блоки **🎙 Заметка преподавателя** — то, что рассказывается вслух
> на занятии. Разворачивайте их при подготовке к защите.

In [ ]:
%%bash
mkdir -p ~/seminar-08/keys ~/seminar-08/ssh ~/seminar-08/logs   # рабочие каталоги семинара
cd ~/seminar-08 || exit 1                                       # дальше каждая ячейка начинается отсюда
ls -F                                                           # пока пусто — наполним по ходу

## 1. Подключение и удалённая команда

SSH создаёт зашифрованное соединение с удалённой машиной. Адрес имеет форму `user@host`; без пользователя берётся текущее локальное имя.

```bash
# Открыть интерактивную сессию.
ssh user@host

# Выполнить одну команду на сервере и завершить соединение.
ssh user@host whoami
ssh user@host 'uname -a'

# Подключиться к нестандартному порту.
ssh -p 2222 user@host
```

`-p` задаёт порт, `-V` показывает версию клиента.

In [ ]:
%%bash
ssh -V      # версия клиента: печатается в stderr, поэтому её видно даже без перенаправления
which ssh   # какой именно бинарник запустится, если набрать ssh

`ssh -G host` **не подключается** к серверу: он печатает итоговые параметры, с которыми клиент собрался бы это делать. Это первый инструмент диагностики — видно, какие пользователь, порт и ключ клиент вывел из ваших настроек, и не нужно ждать таймаута.

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
# -T убирает предупреждение про псевдотерминал: из ноутбука stdin не терминал
ssh -T -G student@server.example > logs/effective-default.conf   # -G считает настройки и в сеть не ходит
head -n 6 logs/effective-default.conf                            # хост, пользователь, порт — то, что клиент решил сам

Дальше — вещь, на которой спотыкаются все. Удалённая команда доезжает до сервера **строкой**, и раскрытие переменных зависит от кавычек:

```bash
ssh course "echo $HOME"   # $HOME раскроет ЛОКАЛЬНАЯ оболочка, на сервер уедет уже готовый путь
ssh course 'echo $HOME'   # на сервер уедет буквальный $HOME, раскроет его удалённая оболочка
```

Эффект видно и без сервера: `bash -c` получает такую же готовую строку, что и `ssh`.

In [ ]:
%%bash
VALUE=local
# двойные кавычки: $VALUE подставится здесь, до запуска дочерней оболочки
bash -c "VALUE=remote; echo \"двойные кавычки: $VALUE\""
# одинарные: строка уходит как есть, $VALUE раскроет уже дочерняя оболочка
bash -c 'VALUE=remote; echo "одинарные кавычки: $VALUE"'

#### ❓ **Вопрос**: Локальный `$HOME` равен `/home/local`, удалённый — `/home/student`. Что выведут `ssh course "echo $HOME"` и `ssh course 'echo $HOME'`?

<details>

<summary><strong>Ответ</strong></summary>

Первая команда передаст уже раскрытый локальный путь `/home/local`. Вторая передаст буквальный `$HOME`, который удалённая оболочка раскроет в `/home/student`. В ячейке выше та же пара кавычек дала `local` и `remote`: в двойных кавычках подставляет вызывающая сторона, в одинарных — вызываемая.

</details>

## 2. Ключи SSH

Для постоянного доступа используют ключи: пароль можно перебирать через сеть, а закрытый ключ серверу не передаётся. Пара состоит из закрытого и открытого ключа. Закрытый ключ остаётся у владельца; открытый добавляется на сервер в `~/.ssh/authorized_keys`. Парольная фраза защищает закрытый файл при краже.

Пара ключей: закрытый остаётся у клиента, открытый уезжает в authorized_keys

На Linux закрытый ключ должен быть недоступен другим пользователям. В курсе используем `chmod 400 private_key`: чтение только владельцу. OpenSSH откажется использовать ключ с избыточно широкими правами.

`ssh-keygen` создаёт пару ключей и показывает fingerprint — короткий отпечаток, по которому ключ можно проверить.

- `-t` — тип ключа;
- `-f` — путь;
- `-C` — комментарий;
- `-N` — парольная фраза;
- `-q` — убрать обычные сообщения;
- `-l -f public_key` — показать fingerprint.

Для нового обычного ключа берут `ed25519`: он компактен, быстро создаётся и не требует выбирать размер ключа.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>


Про остальные типы. RSA не «сломан», но для сопоставимой стойкости ему нужны заметно более длинные ключи; его оставляют для совместимости со старыми системами или требованиями конкретной инфраструктуры. Типы `ed25519-sk` и `ecdsa-sk` работают с аппаратным FIDO-ключом, например YubiKey: закрытая часть физически не покидает железку, и украсть её копированием файла нельзя. DSA устарел, использовать его не следует — из свежих сборок OpenSSH его уже выпилили.

И отдельная оговорка, которую стоит проговорить вслух: эта пара ключей нужна **только для аутентификации** — сервер проверяет, что подключается владелец закрытого ключа. Шифрование самого канала согласуется отдельно и другими алгоритмами, например ChaCha20-Poly1305 или AES-GCM. Студенты часто думают, что трафик шифруется «их ключом», — это не так.


</details>

Заведём учебную пару. Мы кладём её в `~/seminar-08/keys/`, а не в `~/.ssh/`, чтобы демонстрация не перемешалась с вашими настоящими ключами.

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
rm -f ~/seminar-08/keys/course_key ~/seminar-08/keys/course_key.pub   # чистый старт: ssh-keygen не перезапишет молча
ssh-keygen -q -t ed25519 -N '' -C 'course-demo' -f keys/course_key    # -N '': без парольной фразы, чтобы демка не спрашивала
chmod 400 keys/course_key                                             # закрытый ключ — чтение только владельцу

Получилось два файла. Смотреть на них стоит внимательно: именно здесь студенты чаще всего отправляют на сервер не тот файл.

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
ls -l keys/course_key keys/course_key.pub   # -r-------- у закрытого; открытый может читать кто угодно
ssh-keygen -lf keys/course_key.pub          # отпечаток пары: длина, SHA256-хэш, комментарий
cat keys/course_key.pub                     # ровно эта одна строка и уезжает в authorized_keys

#### ❓ **Вопрос**: Какой из двух файлов можно передать на сервер и почему второй передавать нельзя?

<details>

<summary><strong>Ответ</strong></summary>

Передают `course_key.pub` — его содержимое дописывается в `~/.ssh/authorized_keys` на сервере. Закрытый `course_key` остаётся на вашей машине и в курсе получает права `400`: он ни при каком раскладе не отправляется по сети, им только подписывают задачу от сервера. Открытый ключ — одна строка, которую и вывел `cat`; из неё восстановить закрытый нельзя.

</details>

### `ssh-agent`: ключи в текущем окружении

Закрытый ключ с парольной фразой обычно приходится разблокировать при каждом подключении. `ssh-agent` — фоновый процесс, который хранит уже разблокированные ключи в памяти и по запросу выполняет ими аутентификацию. Сам закрытый ключ на сервер не отправляется.

SSH-клиент находит агент по переменной окружения `SSH_AUTH_SOCK`. Поэтому ключи агента доступны текущей оболочке и программам, которые запущены из неё и унаследовали эту переменную. Другая независимая оболочка может быть подключена к другому агенту или не видеть агента совсем.

В графической сессии агент часто запускается автоматически, поэтому сначала проверяют `ssh-add -l`. Если агента нет, его поднимают так:

```bash
eval "$(ssh-agent -s)"
ssh-add ~/.ssh/course_ed25519
ssh-add -l
```

`ssh-agent -s` запускает агент и печатает команды, задающие `SSH_AUTH_SOCK` и `SSH_AGENT_PID`. Конструкция `$(...)` получает этот текст, а `eval` выполняет его в текущей оболочке. Простой запуск `ssh-agent` без `eval` лишь напечатает настройки: текущая оболочка не начнёт использовать новый агент.

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
eval "$(ssh-agent -s)" > /dev/null   # без eval переменные напечатались бы, но не установились
ssh-add keys/course_key              # ключ разблокирован и лежит в памяти агента
ssh-add -l                           # что агент готов предъявлять серверам
ssh-agent -k > /dev/null             # демка кончилась — гасим агент, чтобы процесс не висел

#### ❓ **Вопрос**: Ключ добавили в `ssh-agent`, запущенный вручную из одного терминала. Почему в другом терминале этот ключ может быть недоступен?

<details>

<summary><strong>Ответ</strong></summary>

SSH находит агент по переменной `SSH_AUTH_SOCK`. `eval` в ячейке выше задал её **в этой** оболочке, и унаследуют её только её дочерние процессы. Независимый терминал стартует со своим окружением: там либо другой агент, либо агента нет вовсе, и `ssh-add -l` ответит `Could not open a connection to your authentication agent`.

</details>

Если агент предлагает много ключей, сервер может исчерпать лимит попыток до нужного ключа и разорвать соединение с `Too many authentication failures`. `IdentitiesOnly=yes` заставляет клиент использовать только ключи из `-i` и `IdentityFile`:

```bash
ssh -o IdentitiesOnly=yes -i ~/.ssh/course_ed25519 user@host
```

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>


Есть ещё `ForwardAgent yes`: он разрешает удалённой машине обращаться к вашему локальному агенту, чтобы с сервера ходить дальше, не копируя туда закрытый ключ. Звучит удобно, но включать это стоит только для доверенных промежуточных серверов. Тот, кто получил root на сервере, ключ не прочитает — но, пока вы подключены, сможет через ваш сокет ходить вашим ключом куда угодно. Это регулярно всплывает в разборах взломов: одна скомпрометированная машина в цепочке — и дальше идут уже «легальные» входы. Сегодняшний правильный ответ на эту задачу — `ProxyJump` (раздел 3), а не проброс агента.


</details>

## 3. SSH-config

`~/.ssh/config` хранит параметры подключений — это ровно то, что вы будете сдавать в домашке:

- `Host` — короткое имя;
- `HostName` — адрес;
- `User` — пользователь;
- `Port` — порт;
- `IdentityFile` — закрытый ключ;
- `IdentitiesOnly yes` — не предлагать серверу остальные ключи агента.

Чтобы не портить ваш настоящий `~/.ssh/config`, демонстрация пишет отдельный файл и передаёт его через `ssh -F`.

In [ ]:
%%bash
# учебный конфиг отдельным файлом — ваш ~/.ssh/config мы не трогаем
cat > ~/seminar-08/ssh/config <<'EOF'
Host course
    HostName server.example
    User student
    Port 2222
    IdentityFile ~/seminar-08/keys/course_key
EOF

Теперь одно короткое имя заменяет пользователя, адрес, порт и ключ: вместо `ssh -p 2222 -i ... student@server.example` достаточно `ssh course`. Проверим, что именно клиент вывел из этих строк, — снова без подключения.

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
chmod 600 ssh/config                                        # конфиг тоже читает только владелец
ssh -T -F ssh/config -G course > logs/effective-course.conf # -F: взять этот файл вместо ~/.ssh/config
grep -E '^(hostname|user|port|identityfile) ' logs/effective-course.conf   # четыре строки из конфига доехали

#### ❓ **Вопрос**: Как подключиться с ключом `course_key` без файла настроек и какое поле укажет тот же ключ в `~/.ssh/config`?

<details>

<summary><strong>Ответ</strong></summary>

Без файла настроек — `ssh -i ~/seminar-08/keys/course_key student@server.example`. В конфиге это поле `IdentityFile`: в выводе `ssh -G` выше видно, что клиент подставил именно его. Разница только в том, что конфиг не нужно вспоминать при каждом запуске.

</details>

### `ProxyJump`: вход через bastion

Внутренние серверы обычно не имеют внешнего адреса: наружу смотрит один **bastion** (он же jump-host), а всё остальное доступно только из внутренней сети. Заходить в два приёма — «сначала ssh на bastion, оттуда ssh дальше» — плохо: тогда закрытый ключ приходится держать на bastion, а `scp`, `rsync` и проброс портов работать не будут.

ProxyJump: bastion даёт транспорт, сессия шифруется до target

`ProxyJump` решает это одной строкой конфига: клиент открывает соединение с bastion, просит его пробросить TCP до target и уже поверх этого поднимает вторую SSH-сессию. Для `scp`, `rsync` и туннелей `lab` выглядит как обычный хост.

In [ ]:
%%bash
# дописываем в учебный конфиг второй хост — внутренний, без внешнего адреса
cat >> ~/seminar-08/ssh/config <<'EOF'

Host lab
    HostName 10.0.0.7
    ProxyJump course
EOF

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
# -G по-прежнему ничего не подключает: показывает, что до lab клиент пойдёт через course
ssh -T -F ssh/config -G lab | grep -E '^(hostname|proxyjump) '

#### ❓ **Вопрос**: Чем `ProxyJump course` лучше, чем зайти на `course` и запустить `ssh lab` уже там?

<details>

<summary><strong>Ответ</strong></summary>

При `ProxyJump` вторая сессия шифруется от вашей машины до `lab`, а `course` только пропускает через себя байты: закрытый ключ и парольная фраза на bastion не появляются. Плюс `lab` становится обычным хостом конфига — работают `scp lab:...`, `rsync` и проброс портов, чего при ручном заходе в два шага не получить. В выводе `ssh -G lab` это видно строкой `proxyjump course`.

</details>

## 4. Проверка сервера и диагностика

### Проверка соединения

`ssh -Tvvv user@host true` проверяет соединение и аутентификацию без интерактивного терминала: `-T` отключает псевдотерминал, `-vvv` включает максимальную клиентскую диагностику, `true` сразу завершает удалённую команду.

`-o Name=Value` передаёт одну настройку клиента. `ConnectTimeout=5` ограничивает установку соединения, чтобы недоступный сервер не держал вас минуту.

In [ ]:
%%bash
# нужен доступ к учебному серверу course
cd ~/seminar-08 || exit 1
ssh -Tvvv -o ConnectTimeout=5 course true > logs/connection.out 2> logs/connection-debug.txt
echo "код возврата: $?"   # 0 — вошли и вышли; иначе разбираем лог
grep -E 'Server host key|Authentications that can continue|Offering public key|Authenticated to' logs/connection-debug.txt

Лог `-vvv` длинный, но читается по опорным точкам. Ищите в нём такие строки:

- `Connecting to ... port 22` — резолв имени и TCP прошли;
- `Server host key: ssh-ed25519 SHA256:...` — сервер представился, дальше сверка с `known_hosts`;
- `Offering public key: ...` — какой ключ клиент предложил (тут видно, что подсунулся не тот);
- `Authentications that can continue: publickey,password` — что сервер вообще готов принять;
- `Authenticated to ...` — вход состоялся.

Обрыв между «Offering» и «Authenticated» почти всегда означает, что вашего открытого ключа нет в `authorized_keys` или у файлов неверные права.

### Ключ сервера и `known_hosts`

Пользовательский ключ доказывает серверу, кто подключается. Серверный ключ, или host key, решает обратную задачу: помогает клиенту убедиться, что перед ним нужный сервер. При первом подключении SSH показывает fingerprint сервера, а после подтверждения сохраняет его ключ в `~/.ssh/known_hosts`.

Две встречные проверки: authorized_keys на сервере и known_hosts у клиента

Неожиданная смена ключа может означать переустановку сервера или атаку. Новый fingerprint сначала проверяют по независимому доверенному каналу. Нельзя просто удалить старую запись и согласиться с новой.

`ssh-keyscan host` получает публичный ключ сервера, но не подтверждает его подлинность: он берёт то, что ответила сеть, — ровно то же самое, что подсунул бы и злоумышленник.

`ssh-keygen` здесь не создаёт новый ключ: `ssh-keygen -F host -f file` ищет запись хоста в выбранном `known_hosts`, `ssh-keygen -R host -f file` удаляет её.

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
rm -f ~/seminar-08/ssh/known_hosts                                # чистый старт: иначе записи копятся
ssh-keyscan -t ed25519 localhost >> ssh/known_hosts 2>/dev/null   # sshd есть и на вашей машине — цель под рукой
ssh-keygen -F localhost -f ssh/known_hosts || echo 'sshd не ответил — сделайте то же для своего сервера'

#### ❓ **Вопрос**: SSH сообщает, что ключ знакомого сервера изменился. Почему нельзя сразу сделать `ssh-keygen -R host` и подключиться заново?

<details>

<summary><strong>Ответ</strong></summary>

Изменение ключа означает либо переустановку сервера, либо подмену: между вами и сервером кто-то встал и подставляет свой host key. Удалив запись, вы своими руками согласитесь на нового собеседника. Сначала новый fingerprint подтверждают по независимому каналу — у администратора, в панели облака, по второму маршруту. Обратите внимание: `ssh-keyscan` для этого не годится — как видно из ячейки выше, он просто берёт то, что ответила сеть.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>


Серверные ключи должны быть уникальны. При клонировании виртуалки вместе с ключами новый сервер получает чужую идентичность, и `known_hosts` перестаёт что-либо гарантировать. Современный OpenSSH предупредит, если такой ключ уже записан у вас для другого имени, — но только если запись уже есть.

Историческая иллюстрация: в 2006–2008 годах в Debian из OpenSSL по ошибке выкинули часть источника случайности. Все ключи, сгенерированные на таких системах, оказались из очень маленького набора — их можно было просто перебрать по списку. Чинили это всей индустрией: перевыпуск ключей, чёрные списки скомпрометированных отпечатков в самом OpenSSH. Отсюда привычка не тащить ключи в образы и генерировать host key при первом запуске машины.


</details>

## 5. Передача файлов через `scp`

`scp` копирует файлы через SSH и использует те же ключи и настройки подключения. Удалённый путь записывается как `user@host:path` или, с конфигом, `course:path`.

```bash
scp report.txt course:reports/
scp course:reports/report.txt returned.txt
```

Путь без начального `/` считается относительно домашнего каталога удалённого пользователя. `/var/tmp/report.txt` начинается от корня удалённой системы.

`-r` копирует каталог, `-P 2222` задаёт SSH-порт. Здесь используется заглавная `P`: строчная `-p` имеет другое значение (сохранить время и права).

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
mkdir -p transfer
echo 'course report' > transfer/report.txt   # маленький файл, который отправим в круговой рейс
wc -c transfer/report.txt                    # запомним размер: сверим его после возвращения

Теперь сам рейс: файл уезжает на сервер и возвращается уже под другим именем.

In [ ]:
%%bash
# нужен доступ к учебному серверу course
cd ~/seminar-08 || exit 1
ssh course 'mkdir -p reports'                          # каталог назначения в домашнем каталоге на сервере
scp transfer/report.txt course:reports/                # туда: путь без ведущего / — от домашнего каталога
scp course:reports/report.txt transfer/returned.txt    # и обратно, под другим именем

Файл вернулся — но «вернулся» надо доказать. Размер сравнивают через `wc -c`, содержимое — через `cmp`: он ничего не печатает и возвращает `0`, когда файлы совпадают.

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
wc -c transfer/report.txt transfer/returned.txt   # размеры обязаны совпасть
cmp transfer/report.txt transfer/returned.txt && echo 'содержимое совпало'   # cmp молчит, когда файлы одинаковы

#### ❓ **Вопрос**: Куда попадут файлы из `scp file.txt course:reports/` и `scp file.txt course:/reports/`?

<details>

<summary><strong>Ответ</strong></summary>

Первый — в `reports` внутри домашнего каталога удалённого пользователя (`/home/student/reports/`), как в ячейке выше. Второй — в `/reports` от корня удалённой системы; обычно такого каталога нет и прав на его создание тоже, поэтому команда завершится ошибкой.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>


`scp` — древняя команда, и её протокол долго был «запустить на той стороне `scp -t` и общаться с ним по каналу». Отсюда старые сюрпризы: имена с пробелами и `*` раскрывались удалённой оболочкой, а клиент почти не проверял, те ли файлы ему прислали в ответ. Начиная с OpenSSH 9.0 `scp` под капотом использует протокол SFTP — команда та же, поведение с хитрыми именами стало предсказуемее.

Практический вывод для студентов: `scp` хорош ровно для «отправь один файл». Всё, что сложнее, — `rsync`.


</details>

## 6. Синхронизация через `rsync`

`rsync` сравнивает источник и назначение и передаёт изменения:

```bash
rsync -av data/ course:backup/
```

В записи `course:backup/` локальный `rsync` подключается к `course` по SSH и запускает там второй `rsync`. Отдельный сервер rsync настраивать не нужно, но программа должна быть установлена на обеих машинах. Другой SSH-порт задают через `-e 'ssh -p 2222'`.

`scp` удобен для разовой простой копии. `rsync` удобнее для повторной синхронизации дерева: он сравнивает состояния, передаёт изменения и умеет показывать план, исключать пути и удалять лишнее в назначении.

`-a` сохраняет структуру и метаданные, `-v` показывает действия. `data/` означает содержимое каталога, `data` — сам каталог вместе с именем.

Разберём на двух локальных каталогах — механика та же, что и с удалённым назначением.

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
mkdir -p sync/source sync/copy
echo alpha > sync/source/a.txt
echo beta > sync/source/b.txt
ls sync/source sync/copy   # слева два файла, справа пока пусто

Перед любой синхронизацией с `--delete` смотрят план: `-n` (он же `--dry-run`) ничего не меняет, `-i` расшифровывает каждую строку — что за файл и почему он попал в список.

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
# -n: только показать план; -i: колонка вида >f+++++++++ — новый файл, поедет целиком
rsync -avni --delete --exclude='__pycache__/' sync/source/ sync/copy/

План устраивает — повторяем то же самое без `-n`. И сразу запускаем второй раз: вот ради этого `rsync` и берут.

In [ ]:
%%bash
cd ~/seminar-08 || exit 1
rsync -avi --delete --exclude='__pycache__/' sync/source/ sync/copy/   # первый прогон: файлы поехали
echo '--- тот же запуск ещё раз ---'
rsync -avi --delete --exclude='__pycache__/' sync/source/ sync/copy/   # второй: передавать нечего, список пуст

#### ❓ **Вопрос**: Когда для каталога разумнее выбрать `rsync`, а не `scp`, и чем отличаются источники `data` и `data/`?

<details>

<summary><strong>Ответ</strong></summary>

`rsync` выбирают для повторной синхронизации: во втором прогоне выше он сравнил состояния и не передал ничего, тогда как `scp -r` перелил бы всё заново. `scp` подходит для простой разовой копии одного файла. Про слэш: `data` копирует сам каталог с его именем (получится `backup/data/...`), `data/` — только его содержимое (получится `backup/...`).

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>


`rsync` написали Эндрю Триджелл и Пол Маккеррас, первый релиз — 1996 год. Интересна там не программа, а алгоритм из диссертации Триджелла: получатель режет свой файл на блоки и шлёт для каждого пару контрольных сумм — быструю «скользящую» и криптографическую. Отправитель прогоняет скользящую сумму по своему файлу со сдвигом в один байт и находит совпадения на любых смещениях. По сети уходят только те куски, которых у получателя нет, — даже если файл огромный, а изменилась середина.

Это стоит проговорить вслух: `--delete` без предварительного `-n` — классический способ стереть данные, которые «должны были быть в источнике». Ставьте `-n` рефлекторно.


</details>

## 7. Туннели

Типичная ситуация: Jupyter на сервере слушает `127.0.0.1:8888`, наружу порт не открыт (и правильно — открывать его в интернет нельзя), а работать с ним хочется из браузера на своём ноутбуке. Открывать порт в firewall не нужно: SSH умеет пробрасывать соединения внутри уже установленной зашифрованной сессии.

Пробросов два, и их путают всегда.

Локальный проброс -L и удалённый -R: где открывается слушающий порт

### `-L` — локальный проброс

В `-L 9000:127.0.0.1:8888`:

- `9000` — порт, который откроется **на вашей машине**;
- `127.0.0.1:8888` — адрес сервиса **со стороны SSH-сервера**;
- `course` — сервер, через который идёт соединение.

```bash
ssh -N -o ExitOnForwardFailure=yes \
  -L 9000:127.0.0.1:8888 course
```

`-N` не запускает удалённую команду: соединение нужно только ради проброса. `ExitOnForwardFailure=yes` завершает SSH, если порт открыть не удалось, — иначе легко получить «висящую» сессию без туннеля и долго гадать, почему браузер не отвечает. Пока команда работает, локальный адрес проверяют обычным клиентом сервиса: `curl http://127.0.0.1:9000`.

### `-R` — удалённый проброс

`-R 9000:127.0.0.1:8888 course` открывает порт `9000` **на сервере**, а соединения с него выходят на `127.0.0.1:8888` уже **у вас**. Так показывают коллеге сервис, поднятый на своём ноутбуке, или пускают сервер к тому, до чего он сам дотянуться не может.

Направление самого SSH-соединения при этом не меняется: и в `-L`, и в `-R` клиент — вы, сервер — `course`. Меняется только то, на какой из двух машин появляется слушающий порт.

По умолчанию проброшенный порт слушает на `127.0.0.1`, то есть доступен только с самой машины; открыть его на все интерфейсы разрешают явно (`GatewayPorts` на сервере для `-R`, `-g` или адрес в аргументе для `-L`). Делать это на машине с внешним адресом — значит выставить сервис в интернет, поэтому по умолчанию так и не сделано.

#### ❓ **Вопрос**: Команда `ssh -R 9000:127.0.0.1:3000 course` выполнена на вашем ноутбуке. На какой машине откроется порт 9000 и на какой машине должен работать сервис на порту 3000?

<details>

<summary><strong>Ответ</strong></summary>

Порт `9000` откроется **на сервере** `course` — это и есть смысл `-R`. Сервис на `127.0.0.1:3000` должен работать **на вашем ноутбуке**: адрес после первого двоеточия всегда трактуется с той стороны, которая противоположна слушающему порту. Мнемоника со схемы: первое число — порт, который открывается; `-L` открывает у вас, `-R` — на сервере.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>


Туннель — это дырка в периметре, и админы это понимают. На серверах, где такое не нужно, ставят `AllowTcpForwarding no` в `sshd_config`; обратный проброс `-R` с `GatewayPorts yes` — вообще типовой способ вытащить внутренний сервис наружу в обход firewall, поэтому за ним следят.

Из практики: связка «Jupyter на сервере + `ssh -L`» — стандартный рабочий процесс на любом кластере с GPU, где нет веб-интерфейса. Если туннель постоянно рвётся, помогает `autossh` или `ServerAliveInterval 30` в конфиге. И туннель — не VPN: он пробрасывает конкретный TCP-порт, а не сеть целиком.


</details>

## 8. `tmux`: работа после отключения

Обычная удалённая команда и процессы, привязанные к SSH-терминалу, при разрыве соединения теряют терминал и обычно завершаются. Простой запуск через `&` не гарантирует, что процесс переживёт отключение.

`tmux` запускает на сервере собственную сессию с терминалами. Она продолжает работать и после `Ctrl+B`, затем `D`, и после внезапного разрыва SSH без явного отсоединения.

```bash
# Создать интерактивную сессию.
tmux new -s work

# Вернуться к существующей сессии.
tmux attach -t work
```

Комбинации внутри tmux начинаются с префикса: нажмите `Ctrl+B`, отпустите, затем нажмите следующую клавишу. В краткой записи `C-b` означает `Ctrl+B`.

- `C-b d` — отсоединиться;
- `C-b c` — создать окно;
- `C-b n`, `C-b p`, `C-b 0…9` — перейти между окнами;
- `C-b %` — разделить окно на левую и правую панели;
- `C-b "` — разделить окно на верхнюю и нижнюю панели;
- `C-b` и стрелка — перейти в соседнюю панель;
- `C-b q` — показать номера панелей.

`tmux` одинаково работает и на сервере, и на вашей машине, поэтому следующие две ячейки можно выполнить прямо здесь. `-d` создаёт сессию сразу отсоединённой — то же состояние, в котором она остаётся после `C-b d`.

In [ ]:
%%bash
tmux kill-session -t seminar08 2>/dev/null    # на случай повторного запуска ячейки
tmux new-session -d -s seminar08 'sleep 300'  # -d: сессия создаётся отсоединённой, терминал ей не нужен
tmux list-sessions                            # она есть, хотя мы к ней не подключены

Сессия живёт сама по себе. Заглянем, какой процесс держит её панель, и приберём за собой: `has-session` возвращает `0`, если сессия есть, поэтому его удобно использовать в проверках.

In [ ]:
%%bash
tmux list-panes -t seminar08 -F '#{pane_pid} #{pane_current_command}'   # PID и команда внутри панели
tmux kill-session -t seminar08                                          # закрываем сессию
tmux has-session -t seminar08 2>/dev/null || echo 'сессии больше нет'   # ненулевой код = сессии нет

#### ❓ **Вопрос**: SSH-соединение оборвалось, пока пользователь работал внутри `tmux` и не нажимал detach. Завершится ли tmux-сессия и как к ней вернуться?

<details>

<summary><strong>Ответ</strong></summary>

Сессия продолжит работать на сервере: она принадлежит не вашему SSH-соединению, а серверному процессу `tmux`, — как в ячейке выше, где сессия существовала без единого подключённого терминала. После нового входа на сервер к ней возвращаются через `tmux attach -t NAME`, а список смотрят через `tmux list-sessions`.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>


Предшественник tmux — GNU `screen`, ему больше тридцати лет; tmux появился в 2007 году и постепенно вытеснил его. Если на чужом сервере нет tmux — почти наверняка есть screen, префикс там `C-a`.

Про границы применимости, это стоит сказать вслух. `nohup COMMAND > run.log 2>&1 &` проще, если к экрану возвращаться не нужно: команда отвязывается от терминала, но интерактивности у неё нет. `tmux` — для интерактивных экспериментов и наблюдения за долгим счётом. А вот долго работающий сервис в tmux не держат: им управляет systemd с политикой запуска, перезапуска и журналами. Деплой тоже должен быть воспроизводимым скриптом, а не набором действий в чьём-то сохранённом терминале — «оно работает, только не убивайте мою сессию» звучит смешно ровно до первой перезагрузки сервера.


</details>

## Дополнительно

Этот раздел на занятии не показывают — он нужен при решении задач и подготовке к защите.

### Установка публичного ключа

Первичный доступ обычно выдаёт администратор или облачная платформа. Пока этот доступ действует, публичный ключ можно одной командой добавить в `~/.ssh/authorized_keys`:

```bash
ssh-copy-id -i ~/seminar-08/keys/course_key.pub course
```

`ssh-copy-id` подключается по SSH, создаёт нужные файлы с правильными правами и не добавляет уже установленный ключ повторно. Флаг `-i` выбирает публичный ключ. Новый вход по ключу проверяют во втором терминале, **не закрывая** рабочее или аварийное соединение. Для постоянного доступа к серверу используют ключи, а парольную аутентификацию отключают только после такой проверки.

### Окна и вывод `tmux`

Иерархия tmux: **сессия → окна → панели**. В примере сессия называется `demo`, окна — `clock` и `identity`; в каждом окне пока одна панель.

```bash
tmux new-session -d -s demo -n clock 'date; sleep 60'
tmux new-window -t demo -n identity 'whoami; sleep 60'
tmux list-windows -t demo
tmux capture-pane -p -t demo:clock
```

Команды читаются так:

- `new-session -d -s demo -n clock COMMAND` — создать отсоединённую сессию `demo`, назвать первое окно `clock` и запустить в нём `COMMAND`;
- `new-window -t demo -n identity COMMAND` — добавить в выбранную через `-t` сессию окно `identity`;
- `list-windows -t demo` — показать окна этой сессии;
- `capture-pane -p -t demo:clock` — вывести содержимое панели окна `clock` в stdout. В адресе `session:window` двоеточие отделяет имя сессии от имени окна.

`sleep 60` оставляет окно живым после завершения первой команды. Без продолжающегося процесса окно сразу закроется.

### Серверная сторона: `sshd`

`ssh` — клиент, а `sshd` — серверный процесс, который принимает соединения. Его настройки находятся в `/etc/ssh/sshd_config` и дополнительных файлах настроек дистрибутива.

Ошибочная настройка может отрезать удалённый доступ. Перед применением конфигурацию проверяют через `sudo sshd -t`, текущую сессию не закрывают, а новое подключение проверяют во втором терминале. После проверки конфигурацию перечитывает служба `ssh` или `sshd` — имя зависит от дистрибутива. Если все способы входа уже сломаны, нужен аварийный режим облака, последовательная консоль или другой независимый доступ.

### Журналы SSH и `fail2ban`

Открытый в интернет SSH-сервер постоянно получает автоматические попытки входа по популярным именам и паролям. Живой журнал обычно смотрят одной из команд:

```bash
sudo journalctl -u ssh -f
sudo journalctl -u sshd -f
```

Имя службы зависит от дистрибутива. Строки вида `Failed password for invalid user ...` показывают не «целевую атаку», а обычный сетевой шум. Поэтому постоянный вход настраивают по ключам, а не по паролю.

`fail2ban` читает журналы и временно блокирует адреса после серии неудачных попыток. Настройки не правят в `jail.conf` — его перезаписывает обновление пакета; свои значения кладут в `/etc/fail2ban/jail.local`:

```ini
[sshd]
enabled  = true
maxretry = 5
findtime = 10m
bantime  = 1h
```

Проверяют результат так:

```bash
sudo fail2ban-client status          # какие jail включены
sudo fail2ban-client status sshd     # сколько адресов забанено прямо сейчас
sudo fail2ban-client set sshd unbanip 203.0.113.10   # разбанить (в том числе себя)
```

Это дополнительный барьер, а не замена ключам, обновлениям и сетевым ограничениям.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>


Порядок цифр, чтобы студенты понимали масштаб: на свежей виртуалке с публичным адресом и SSH на 22-м порту счёт неудачных попыток идёт на тысячи в сутки, и начинаются они через считанные минуты после того, как адрес засветился. Это фоновое сканирование всего IPv4, никакого отношения лично к вам оно не имеет.

Отсюда два вывода. Первый: перенос порта на 2222 убирает шум из логов, но не является защитой — сканеры давно ходят по всем портам. Второй: `fail2ban` защищает от перебора, а не от компрометации ключа или дыры в сервисе. Реальная защита — вход только по ключам, `PasswordAuthentication no` и обновления. И почти обязательный ритуал: рано или поздно вы забаните сами себя со своего офисного адреса — поэтому `unbanip` стоит помнить наизусть, а на облачной машине держать аварийный доступ через консоль провайдера.


</details>